In [1]:
import pandas as pd

df = pd.read_csv("news.csv")

print(df.head())

                                       text  label 
0  Government launches new education policy       1
1         Scientists discover water on Mars       1
2            Aliens attack Mumbai yesterday       0
3              Drinking petrol cures cancer       0
4              India wins cricket world cup       1


In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

In [3]:
sample = df['text'][0]

tokens = tokenizer(
    sample,
    padding='max_length',
    truncation=True,
    max_length=32,
    return_tensors='pt'
)

print(tokens)

{'input_ids': tensor([[  101,  2231, 18989,  2047,  2495,  3343,   102,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]])}


In [5]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1757.99it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [6]:
from torch.optim import AdamW
import torch

optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()

for epoch in range(3):

    outputs = model(
        input_ids=tokens['input_ids'],
        attention_mask=tokens['attention_mask'],
        labels=torch.tensor([1])
    )

    loss = outputs.loss

    loss.backward()

    optimizer.step()

    optimizer.zero_grad()

    print("Loss:", loss.item())

Loss: 0.37897685170173645
Loss: 0.40502840280532837
Loss: 0.24734897911548615


In [7]:
test_news = "Aliens landed in Delhi"

inputs = tokenizer(
    test_news,
    return_tensors="pt",
    truncation=True,
    padding=True
)

model.eval()

with torch.no_grad():

    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits)

print(prediction.item())

1


In [8]:
from sklearn.metrics import accuracy_score

y_true = [1,1,0,0]

y_pred = [1,1,0,1]

acc = accuracy_score(y_true,y_pred)

print("Accuracy =",acc)

Accuracy = 0.75


In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true,y_pred)

print(cm)

[[1 1]
 [0 2]]
